In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import sys
import random
import os
from dotenv import load_dotenv
# Load environment variables from .env file
load_dotenv()
resum_path = os.getenv("RESUM_PATH")
if resum_path is None:
    raise ValueError("Environment variable RESUM_PATH is not set. Make sure to define it in your .env file.")
utilities_path = os.path.join(resum_path, "utilities")
sys.path.append(utilities_path)
import utilities as utils


In [ ]:
iter=3
version="v1.4"
for no in range(300,309):
    path = f'{resum_path}/out/LF/{version}/tier2/neutron-sim-LF-{version}-{no:04d}-tier2_{iter:04d}.csv'
    if not os.path.exists(path):
        continue
    data_train = pd.read_csv(path,index_col=0)
    path2 = f'{resum_path}/adaptive_importance_sampling/out/LF/{version}/neutron-inputs-{version}/neutron-inputs2-design0_10000_{no:04d}_{iter:04d}_{version}.dat'
    data_input = pd.read_csv(path2,sep=" ",index_col=0)
    weights=data_input["weights"].to_numpy()
    data_train['weights']=["{:.5e}".format(val) for val in weights]
    #print(data_train)
    data_train.to_csv(path)

In [ ]:
iter=3
version="v1.4"
for no in range(300,309):
    path = f'{resum_path}/out/LF/{version}/tier2/neutron-sim-LF-{version}-{no:04d}-tier2_'
    file_list=utils.get_all
    if not os.path.exists(path):
        continue
    data_train = pd.read_csv(path,index_col=0)
    
    path2 = f'{resum_path}/adaptive_importance_sampling/out/LF/{version}/neutron-inputs-{version}/neutron-inputs2-design0_10000_{no:04d}_{iter:04d}_{version}.dat'
    data_input = pd.read_csv(path2,sep=" ",index_col=0)
    weights=data_input["weights"].to_numpy()
    data_train['weights']=["{:.5e}".format(val) for val in weights]
    #print(data_train)
    data_train.to_csv(path)

In [ ]:
for i in range(300,310):
    path = f'./out/LF/{version}/tier2/neutron-sim-LF-{version}-{i:04d}-tier2_'
        
    file_list = utils.get_all_files(path,".csv")
    n_files = len(file_list)

    random_rows=False
    n_samples = 20000
    # Initialize an empty DataFrame
    combined_df = pd.DataFrame()

    for file_in in tqdm(file_list):
        # Construct the file name
        # Read the current file
        df = pd.read_csv(file_in)
        total_rows = sum(1 for _ in open(file_in)) - 1
        if random_rows==True:
            # Total number of rows in the file (excluding the header row)
            # Get total rows in the file
            # Randomly select row indices to read (excluding the header row)
            rows_to_read = sorted(random.sample(range(1, total_rows + 1), n_samples))
            df = pd.read_csv(file_in, skiprows=lambda x: x not in rows_to_read and x > 0, index_col=0)
        else:
            if total_rows > n_samples:
                df = pd.read_csv(file_in, index_col=0, nrows=n_samples)
            else:
                df = pd.read_csv(file_in, index_col=0)
        # Append the DataFrame to combined_df
        combined_df = pd.concat([combined_df, df], ignore_index=True)
    # Print the combined DataFrame
    print(combined_df)
    combined_df.to_csv(f'./out/LF/{version}/tier4/neutron-sim-LF-{version}-{i:04d}-tier4.csv')

In [ ]:
version='v1.4xmin'
iter=0
for i in range(6,10):
    path = f'/Users/aschuetz/Documents/Analysis/legend/ML/legend-multi-fidelity-surrogate-model/simulation/out/LF/n1.4/tier2/neutron-sim-LF-{version}-{i:04d}-tier2.csv'
    if not os.path.exists(path):
        continue
    data_train = pd.read_csv(path, index_col=0)
    weights_new=[1./len(data_train) for i in range(len(data_train))]
    data_train['weights']=["{:.3e}".format(val) for val in weights_new]
    data_train["nprimaries"]=1.
    #print(data_train[:10])
    path_new = f'/Users/aschuetz/Documents/Analysis/legend/ML/legend-multi-fidelity-surrogate-model/simulation/out/LF/{version}/tier2/neutron-sim-LF-{version}-{i:04d}-tier2_{iter:04d}.csv'
    data_train.to_csv(path)


    # Set parameter name/x_labels -> needs to be consistent with data input file
    #x_labels=["x_0[m]","y_0[m]","z_0[m]","px_0[m]","py_0[m]","pz_0[m]","ekin_0[eV]"]
    #x_labels=["radius","thickness","npanels","theta","length"]
    #y_label = 'nC_Ge77'

    #x_lf_sig = data_train[(data_train[y_label] == 1)][x_labels].to_numpy()
    #print(f"{i} Total rows with y(x) = 1: {x_lf_sig.shape[0]}/{len(data_train)}")
    #
    #os.rename(path, path_new)

In [31]:
for i in range(300,310):
    boosted=0
    for j in range(0,3):
        filename_base=f"/Users/aschuetz/Documents/Analysis/legend/ML/legend-multi-fidelity-surrogate-model/simulation/out/LF/v1.4/tier2/neutron-sim-LF-v1.4-{i:04d}-tier2_{j:04d}"
        tmp=utils.get_all_files(filename_base)
        data_train = pd.read_csv(tmp[0])
        x_labels=["x_0[m]","y_0[m]","z_0[m]","px_0[m]","py_0[m]","pz_0[m]","ekin_0[eV]"]
        y_label = 'nC_Ge77'
        # Find rows where y_label == 1
        row_lf_sig = data_train.index[data_train[y_label] == 1]
        # Extract corresponding rows and append to the list
        #print(i,j,len(data_train.loc[row_lf_sig][x_labels].to_numpy()), len(data_train))
        if j==0:
            non_boosted=len(data_train.loc[row_lf_sig][x_labels].to_numpy())
        else:
            boosted+=len(data_train.loc[row_lf_sig][x_labels].to_numpy())
    print(i,non_boosted,boosted)

300 12 11
301 4 36
302 9 1
303 15 0
304 10 1
305 7 6
306 8 2
307 7 6
308 6 10
309 5 8


In [ ]:
version='v1.4'
iter=0
for i in range(0,300):
    path = f'/Users/aschuetz/Documents/Analysis/legend/ML/legend-multi-fidelity-surrogate-model/simulation/out/LF/{version}/tier2/neutron-sim-LF-{version}-{i:04d}-tier2.csv'
    if not os.path.exists(path):
        continue
    data_train = pd.read_csv(path, index_col=0, nrows=20000)
    weights_new=[1./len(data_train) for i in range(len(data_train))]
    data_train['weights']=["{:.3e}".format(val) for val in weights_new]
    data_train["nprimaries"]=1.
    #print(data_train[:10])
    path_new = f'/Users/aschuetz/Documents/Analysis/legend/ML/legend-multi-fidelity-surrogate-model/simulation/out/LF/{version}/tier2/neutron-sim-LF-{version}-{i:04d}-tier2_{iter:04d}.csv'
    data_train.to_csv(path_new)


    # Set parameter name/x_labels -> needs to be consistent with data input file
    #x_labels=["x_0[m]","y_0[m]","z_0[m]","px_0[m]","py_0[m]","pz_0[m]","ekin_0[eV]"]
    #x_labels=["radius","thickness","npanels","theta","length"]
    #y_label = 'nC_Ge77'

    #x_lf_sig = data_train[(data_train[y_label] == 1)][x_labels].to_numpy()
    #print(f"{i} Total rows with y(x) = 1: {x_lf_sig.shape[0]}/{len(data_train)}")
    #
    #os.rename(path, path_new)

In [6]:
df=pd.read_csv("/Users/aschuetz/Documents/Analysis/legend/ML/legend-multi-fidelity-surrogate-model/simulation-analysis/out/n1.4/Ge77_rates_n1.4.csv")
x_labels=["y_raw(LF)","y_raw(LF AIS)","y_raw(HF ground truth)"]

In [7]:
df=df[x_labels]

In [6]:
import matplotlib.pyplot as plt

In [ ]:
x=[i for i in range(10)]
rLF=df["y_raw(LF)"].to_numpy()
rAIS=df["y_raw(LF AIS)"].to_numpy()
rHF=df["y_raw(HF ground truth)"].to_numpy()

MSE_LF=(rHF-rLF)**2
MSE_AIS=(rHF-rAIS)**2

#df["MSE_LF"]=MSE_LF
#df["MSE_AIS"]=MSE_AIS



df.to_csv("/Users/aschuetz/Documents/Analysis/legend/ML/legend-multi-fidelity-surrogate-model/simulation-analysis/out/n1.4/Ge77_rates_n1.4.csv")

In [12]:
aver_LF=np.mean(MSE_LF)
aver_AIS=np.mean(MSE_AIS)
print(aver_LF,aver_AIS)

0.0010064329000000006 0.0034462197700000003


In [10]:
df

,y_raw(LF),y_raw(LF AIS),y_raw(HF ground truth),MSE_LF,MSE_AIS
0,0.32124,0.32051,0.23795,0.006937,0.006816
1,0.13002,0.21761,0.15294,0.000525,0.004182
2,0.14980,0.24280,0.12578,0.000577,0.013694
3,0.23298,0.25374,0.24089,0.000063,0.000165
4,0.13920,0.22771,0.14470,0.000030,0.006891
5,0.15960,0.21050,0.16710,0.000056,0.001884
6,0.18560,0.19835,0.19390,0.000069,0.000020
7,0.15160,0.21796,0.18970,0.001452,0.000799
8,0.16800,0.17575,0.17780,0.000096,0.000004
9,0.14080,0.15976,0.15690,0.000259,0.000008


In [ ]:
version_in='v1.4'

for i in range(1,300):
    n_LF=i
    path = f'{resum_path}/simulation/out/LF/{version_in}/tier2/neutron-sim-LF-{version_in}-{n_LF:04}-tier2_'
    data_train, sample_iter = utils.get_dataframes_concat(path, None)
    #print(len(data_train))
    #print(data_train.head())
    #data_train.to_csv(f'{resum_path}/simulation/out/LF/{version_in}/tier4/neutron-sim-LF-{version_in}-{n_LF:04}-tier4.csv')

100%|██████████| 4/4 [00:00<00:00, 11.60it/s]


5
0
0
0


100%|██████████| 4/4 [00:00<00:00, 13.36it/s]


11
1
0
0


100%|██████████| 4/4 [00:00<00:00, 11.19it/s]


...

100%|██████████| 4/4 [00:00<00:00, 14.20it/s]


12
0
0
0


 50%|█████     | 2/4 [00:00<00:00, 10.22it/s]


6
4


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.